In [1]:
import pandas as pd
import numpy as np

from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LinearRegression, Lasso, LassoCV

from sklearn.metrics import r2_score, mean_squared_error

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

In [2]:
def assign_age_group(age):
    """
    Assigns a respondent to an age group based on their age.

    Parameters:
        age (int or float): The respondent's age.

    Returns:
        str: The age group label used for retirement readiness analysis.
    """
    if age < 30:
        return "Under 30"
    elif age < 40:
        return "30-39"
    elif age < 50:
        return "40-49"
    elif age < 60:
        return "50-59"
    elif age < 65:
        return "60-64"
    else:
        return "65+"

def retirement_multiplier(age):
    """
    Returns the retirement savings multiplier associated with a respondent's age.

    The multiplier represents the estimated amount of retirement liquidity
    a respondent should have saved as a multiple of annual income for their
    age group.

    Parameters:
        age (int or float): The respondent's age.

    Returns:
        float: The retirement readiness multiplier for the respondent's age group.
    """
    if age < 30:
        return 0.5
    elif age < 40:
        return 1
    elif age < 50:
        return 3
    elif age < 60:
        return 6
    elif age < 65:
        return 8
    else:
        return 10

In [3]:
df = pd.read_csv('../data/processed/Cleaned_SCF_2022.csv', index_col=0)
df.shape

(4595, 12)

In [4]:
df.head()

,LOG_RETQLIQ,EDUC,HHOUSES,EMERGSAV,FINLIT,AGE,LOG_EQUITY,LOG_FIN,LOG_LIQ,LOG_INCOME,LOG_DEBT,LOG_STOCKS
0,12.409018,9,1,1,1,70,10.799596,12.461106,9.480444,10.566323,12.180760,0.000000
5,11.512935,12,1,1,3,46,10.308986,11.852970,10.609082,12.323103,13.159083,0.000000
10,13.036808,14,1,1,3,68,17.917856,17.918036,9.295692,13.006586,16.682273,17.904508
15,14.220309,12,0,1,3,74,14.360158,14.587007,12.007628,11.934327,0.000000,0.000000
20,0.000000,8,0,0,0,19,0.000000,7.003974,7.003974,10.936822,9.305741,0.000000


In [5]:
df.columns

Index(['LOG_RETQLIQ', 'EDUC', 'HHOUSES', 'EMERGSAV', 'FINLIT', 'AGE',
       'LOG_EQUITY', 'LOG_FIN', 'LOG_LIQ', 'LOG_INCOME', 'LOG_DEBT',
       'LOG_STOCKS'],
      dtype='str')

verify age group and multiplier are working as expected

In [6]:
df["AGE_GROUP"] = df["AGE"].apply(assign_age_group)
df["RET_MULTIPLIER"] = df["AGE"].apply(retirement_multiplier)

df[["AGE", "AGE_GROUP", "RET_MULTIPLIER"]].head()

,AGE,AGE_GROUP,RET_MULTIPLIER
0,70,65+,10.0
5,46,40-49,3.0
10,68,65+,10.0
15,74,65+,10.0
20,19,Under 30,0.5


create classification target

In [7]:
df["INCOME_RAW_EST"] = np.expm1(df["LOG_INCOME"])
df["RETQLIQ_RAW_EST"] = np.expm1(df["LOG_RETQLIQ"])

df["RET_READY_THRESHOLD"] = df["INCOME_RAW_EST"] * df["RET_MULTIPLIER"]

df["RET_READY"] = np.where(
    df["RETQLIQ_RAW_EST"] >= df["RET_READY_THRESHOLD"],
    1,
    0
)

verify classification is working

In [8]:
df[[
    "AGE",
    "AGE_GROUP",
    "LOG_INCOME",
    "INCOME_RAW_EST",
    "RET_MULTIPLIER",
    "RET_READY_THRESHOLD",
    "LOG_RETQLIQ",
    "RETQLIQ_RAW_EST",
    "RET_READY"
]].head()

,AGE,AGE_GROUP,LOG_INCOME,INCOME_RAW_EST,RET_MULTIPLIER,RET_READY_THRESHOLD,LOG_RETQLIQ,RETQLIQ_RAW_EST,RET_READY
0,70,65+,10.566323,38804.734469,10.0,3.880473e+05,12.409018,245000.0,0
5,46,40-49,12.323103,224829.659320,3.0,6.744890e+05,11.512935,100000.0,0
10,68,65+,13.006586,445335.671340,10.0,4.453357e+06,13.036808,459000.0,0
15,74,65+,11.934327,152408.567130,10.0,1.524086e+06,14.220309,1499000.0,0
20,19,Under 30,10.936822,56207.414830,0.5,2.810371e+04,0.000000,0.0,0


In [9]:
df["RET_READY"].value_counts()

RET_READY
0    4342
1     253
Name: count, dtype: int64

In [10]:
df["RET_READY"].value_counts(normalize=True).round(3)

RET_READY
0    0.945
1    0.055
Name: proportion, dtype: float64

The age-adjusted retirement readiness target classified approximately 5.5% of respondents as retirement ready and 94.5% as not retirement ready. This indicates that the classification target is highly imbalanced, so later classification models should be evaluated using precision, recall, F1-score, and possibly class-weighted models instead of relying on accuracy alone.

## Baseline Model

In [ ]:
clf_features = [
    # financial
    'LOG_EQUITY',
    'LOG_FIN',
    'LOG_LIQ',
    'LOG_INCOME',
    'LOG_DEBT',
    'LOG_STOCKS',
    # socioeconomic 
    'EDUC',
    'FINLIT',
    'AGE'
]

target = 'LOG_RETQLIQ'

In [ ]:
X = df[clf_features]
y = df[target]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)

In [ ]:
y_pred = lr.predict(X_test)

In [ ]:
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("Baseline Linear Regression")
print("R²:", r2)
print("RMSE:", rmse)

In [ ]:
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LinearRegression())
])

pipeline.fit(X_train, y_train)
coefs = pipeline.named_steps['lr'].coef_.ravel()
coefficients = pd.Series(coefs, index=X.columns).sort_values(ascending=False)
coefficients

### Lasso  

In [ ]:
lasso_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('lasso', Lasso(alpha=0.01, random_state=42))
])

In [ ]:
lasso_pipeline.fit(X_train, y_train)

In [ ]:
y_pred_lasso = lasso_pipeline.predict(X_test)

In [ ]:
r2_lasso = r2_score(y_test, y_pred_lasso)
rmse_lasso = np.sqrt(mean_squared_error(y_test, y_pred_lasso))

print("Lasso Regression")
print("R²:", r2_lasso)
print("RMSE:", rmse_lasso)

In [ ]:
coefs = lasso_pipeline.named_steps['lasso'].coef_

lasso_coefficients = pd.Series(coefs, index=X.columns).sort_values(ascending=False)
print(lasso_coefficients)

In [ ]:
lasso_cv = Pipeline([
    ('scaler', StandardScaler()),
    ('lasso', LassoCV(cv=5, random_state=42))
])

lasso_cv.fit(X_train, y_train)

best_alpha = lasso_cv.named_steps['lasso'].alpha_
print("Best alpha:", best_alpha)

### Simplify Model

In [ ]:
reduced_features = [
    'LOG_EQUITY',
    'LOG_FIN',
    'LOG_LIQ',
    'LOG_DEBT',
    'LOG_STOCKS',
    'EDUC',
    'FINLIT'
]

X_reduced = df[reduced_features]

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_reduced, y, test_size=0.2, random_state=42
)

lr_reduced = LinearRegression()
lr_reduced.fit(X_train_r, y_train_r)

y_pred_r = lr_reduced.predict(X_test_r)

print("Reduced Model")
print("R²:", r2_score(y_test_r, y_pred_r))
print("RMSE:", np.sqrt(mean_squared_error(y_test_r, y_pred_r)))

## Classification

In [ ]:
median_salary_2022 = 55_058
threshold_65 = median_salary_2022 * 10 

In [ ]:
df['AGE_GROUP'] = pd.cut(df['AGE'], bins=[20,30,40,50,60,70,100])

In [ ]:
# Binary classification target
# 1 = above median retirement liquidity
# 0 = at or below median retirement liquidity

ready_threshold = df['LOG_RETQLIQ'].median()

df['RET_READY'] = (df['LOG_RETQLIQ'] > ready_threshold).astype(int)

df['RET_READY'].value_counts(normalize=True)

In [ ]:
classification_features = [
    'LOG_EQUITY',
    'LOG_FIN',
    'LOG_LIQ',
    'LOG_DEBT',
    'LOG_STOCKS',
    'EDUC',
    'FINLIT',
    'AGE'
]

X_class = df[classification_features]
y_class = df['RET_READY']

In [ ]:
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_class,
    y_class,
    test_size=0.2,
    random_state=42,
    stratify=y_class
)

X_train_c.shape, X_test_c.shape, y_train_c.shape, y_test_c.shape

Logistic Regression Classifier

In [ ]:
log_clf = Pipeline([
    ('scaler', StandardScaler()),
    ('logreg', LogisticRegression(max_iter=1000, random_state=42))
])

log_clf.fit(X_train_c, y_train_c)

y_pred_log = log_clf.predict(X_test_c)

In [ ]:
print("Logistic Regression Classification Model")
print("Accuracy:", accuracy_score(y_test_c, y_pred_log))
print("Precision:", precision_score(y_test_c, y_pred_log))
print("Recall:", recall_score(y_test_c, y_pred_log))
print("F1 Score:", f1_score(y_test_c, y_pred_log))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test_c, y_pred_log))

print("\nClassification Report:")
print(classification_report(y_test_c, y_pred_log))

lr coefficients

In [ ]:
log_coefs = log_clf.named_steps['logreg'].coef_.ravel()

log_coefficients = pd.Series(
    log_coefs,
    index=X_class.columns
).sort_values(ascending=False)

log_coefficients

Random Forest Classifier

In [ ]:
rf_clf = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    max_depth=None
)

rf_clf.fit(X_train_c, y_train_c)

y_pred_rf = rf_clf.predict(X_test_c)

Eval Random Forest Classifier

In [ ]:
print("Random Forest Classification Model")
print("Accuracy:", accuracy_score(y_test_c, y_pred_rf))
print("Precision:", precision_score(y_test_c, y_pred_rf))
print("Recall:", recall_score(y_test_c, y_pred_rf))
print("F1 Score:", f1_score(y_test_c, y_pred_rf))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test_c, y_pred_rf))

print("\nClassification Report:")
print(classification_report(y_test_c, y_pred_rf))

AUC / ROC

In [ ]:
y_prob_log = log_clf.predict_proba(X_test_c)[:, 1]
y_prob_rf = rf_clf.predict_proba(X_test_c)[:, 1]

# ROC values
fpr_log, tpr_log, _ = roc_curve(y_test_c, y_prob_log)
fpr_rf, tpr_rf, _ = roc_curve(y_test_c, y_prob_rf)

# AUC scores
auc_log = auc(fpr_log, tpr_log)
auc_rf = auc(fpr_rf, tpr_rf)

# Plot
plt.figure()
plt.plot(fpr_log, tpr_log, label=f'Logistic Regression (AUC = {auc_log:.3f})')
plt.plot(fpr_rf, tpr_rf, label=f'Random Forest (AUC = {auc_rf:.3f})')
plt.plot([0, 1], [0, 1], linestyle='--')  # baseline

plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - Retirement Readiness Classification')
plt.legend()
plt.show()

Feature importance

In [ ]:
importance = pd.Series(
    rf_clf.feature_importances_,
    index=X_class.columns
).sort_values(ascending=True)

plt.figure()
importance.plot(kind='barh')

plt.title('Feature Importance - Random Forest Classifier')
plt.xlabel('Importance Score')
plt.ylabel('Features')

plt.show()

Financial Readiness by Age group

In [ ]:
df_results = X_test_c.copy()
df_results['AGE'] = df.loc[X_test_c.index, 'AGE']
df_results['AGE_GROUP'] = df.loc[X_test_c.index, 'AGE_GROUP']
df_results['ACTUAL'] = y_test_c
df_results['PREDICTED'] = y_pred_rf

In [ ]:
df_results.groupby('AGE_GROUP')['PREDICTED'].mean()

In [ ]:
df_results.groupby('AGE_GROUP')['PREDICTED'].mean().plot(kind='bar')
plt.title('Predicted Retirement Readiness by Age Group')
plt.ylabel('% Ready')
plt.show()

In [ ]:
for group in df['AGE_GROUP'].unique():
    subset = df[df['AGE_GROUP'] == group]
    
    if subset.shape[0] == 0:
        continue 
    
    X_sub = subset[clf_features]
    y_sub = subset['LOG_RETQLIQ']
    
    model = LinearRegression().fit(X_sub, y_sub)
    
    print(group)
    print(pd.Series(model.coef_, index=clf_features))